# 11.6 對齊和 SFT 能合併嗎？


一種訓練方案先用圖片描述連接視覺與語言，再用問答教任務；另一種直接用問答，同時學習接頭與應答。前者像先學字詞再讀問句，後者像在完整對話中一起學。哪種更好要看基礎能力與資料，不能讓前者多花一倍訓練量後只說順序比較優越。

前置是[11.3描述對齊](https://birdhackor.github.io/tiny-perceptron-vlm/11.3.html)、[11.4圖片問答](https://birdhackor.github.io/tiny-perceptron-vlm/11.4.html)與[7.3有效學習位置](https://birdhackor.github.io/tiny-perceptron-vlm/7.3.html)。SFT是使用人工目標回答的有監督微調；描述對齊也可使用相同文字預測代價，只是任務目標與可訓練範圍不同。總預算要把兩階段都加進去，不能把第一階段當免費準備。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
budget = 10000
plans = {
    "兩階段": {"描述對齊": 4000, "問答微調": 6000},
    "直接問答": {"描述對齊": 0, "問答微調": 10000},
}
for name, plan in plans.items():
    total = sum(plan.values())
    print(name, plan, "總有效位置", total, "符合預算", total == budget)

輸入是一張計劃表，數字表示參與目標代價的回答位置數量，不是樣本筆數。`values()`取兩個階段的數量，`sum()`合計。輸出兩方案總數都是10000、符合預算True。兩階段分4000與6000，直接方案全給問答。這個程式沒有模型，沒有收斂曲線，完成的是實驗前的公平預算核對。

若描述平均短、問答平均長，相同步數不一定有相同位置數量；應由實際標籤中非-100的數量計費，如[8.3](https://birdhackor.github.io/tiny-perceptron-vlm/8.3.html)的例子。也可以固定總執行時間比較，但要報告每方案實際用了多少有效位置與哪些參數，因為不同方案計算成本可能不同。

兩方案還要從同一份初始模型出發，結束後回答同一批未參與訓練的圖片題，例如給一張保留的方塊圖，問「方塊是什麼顏色？」，用該圖的正確顏色核對。這批題目檢查新學的圖片問答能力。

另外共用一批純文字題，例如「只回答yes」，目標答案是`yes`，不提供圖片。先用訓練前模型回答這批題並記錄正確率，再對照兩方案微調後的正確率，觀察原有的純文字回答能力是否退步；這就是本節的「文字保留」。保留檢查要包含基礎模型原本能答對的題目，才看得出學新能力後是否仍會做。兩類評估題都留在訓練之外，兩方案使用相同題目與判分規則。

兩階段通常把第一階段接頭權重帶進第二階段；直接方案沒有這一步，但其全部10000位置都在問答學習。它們比較的是預算分配與訓練路徑，不只是把同一批資料文件換個名字。上面的預算表沒有執行這些評估，能力變化仍須在實際訓練後測量。

兩階段還涉及交接內容。第一階段產生更新後的接頭，第二階段應從這份狀態繼續，而不是誤從隨機接頭重新開始；否則你花了4000位置卻沒有把學到的東西帶過去。另一方面，第二階段若開放語言層，需在記錄中注明與第一階段不同的範圍。比較時保存每階段起點、終點和有效位置計數，就能重現所謂先對齊再問答的含義。直接問答方案則只有一個起點與一個階段，它可能也通過問答學會對齊，但要由答案測試證明。

練習把對齊預算4000改2000，先預測問答需從6000改8000才能維持10000，再修改核對True。若只改第一項就輸出總數8000、False，計劃已不再匹配。提出「對齊佔20%可能夠嗎」是可測試假設，不能從這張預算表直接讀出答案。
